<a href="https://colab.research.google.com/github/alexcho2823-del/ds2002-fa26/blob/main/05-etl-walmart/2026-10-16%20%E2%80%94%20Walmart%20Warm-Up%20SKU%20Mess%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '/content/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))

pop_mask = cat['product_name'].str.lower().str.contains('pop', na=False)
pop = cat[pop_mask]

print(f"Rows containing 'pop': {len(pop)}")
pop

Rows containing 'pop': 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


Lowercased every product name, flagged the ones containing “pop” (na=False skips blank names instead of crashing), and printed the count and matching rows.

### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
# TODO
name = cat['product_name'].str.lower()

is_poptart = name.str.contains('pop', na=False) & name.str.contains('tart', na=False)
is_straw   = name.str.contains('straw', na=False)

straw = cat[is_poptart & is_straw]
print("=== Strawberry Pop-Tart variants ===")
print(straw[['sku', 'product_name', 'category']].to_string())

not_straw = cat[pop_mask & ~(is_poptart & is_straw)]
print("\n=== 'pop' rows that are NOT strawberry Pop-Tarts ===")
print(not_straw[['sku', 'product_name', 'category']].to_string())

=== Strawberry Pop-Tart variants ===
             sku          product_name         category
0          PT-12  pop-tarts strawberry  Breakfast Foods
1     SKU#459812   Strawberry Pop-Tart        Breakfast
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast
3  PT_STRAWBERRY  pop-tarts strawberry        breakfast

=== 'pop' rows that are NOT strawberry Pop-Tarts ===
Empty DataFrame
Columns: [sku, product_name, category]
Index: []


**Pop-Tart rows that are a different product:** None. All four “pop” rows (PT-12, SKU#459812, POPTART-STRAW, PT_STRAWBERRY) are strawberry Pop-Tarts. If other flavors appeared, they would stay as separate SKUs.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
# TODO
n_rows_original = len(cat)

canonical = straw['sku'].iloc[0]
sku_map = {s: canonical for s in straw['sku'] if s != canonical}
print("Canonical SKU:", canonical)
print("Mapping:", sku_map)

cat['canonical_sku'] = cat['sku'].replace(sku_map)
cat[cat['sku'] != cat['canonical_sku']]

Canonical SKU: PT-12
Mapping: {'SKU#459812': 'PT-12', 'POPTART-STRAW': 'PT-12', 'PT_STRAWBERRY': 'PT-12'}


,sku,product_name,category,subcategory,unit_cost,supplier,package_size,canonical_sku
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk,PT-12
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack,PT-12
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack,PT-12


Picked one strawberry SKU as the official one, built a dict sending every other variant to it, and applied it with .replace(), which leaves unmapped SKUs unchanged.

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
# TODO
print("=== BEFORE ===")
print(cat['category'].value_counts(dropna=False))

cat['category_clean'] = cat['category'].str.strip().str.title()

print("\n=== AFTER strip + title-case ===")
print(cat['category_clean'].value_counts(dropna=False))

=== BEFORE ===
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64

=== AFTER strip + title-case ===
category_clean
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
H

Showed the raw category counts, removed extra spaces and fixed capitalization, and showed the counts again so you can see which labels collapsed together.

Still duplicated after strip/title-case: Breakfast / Breakfast Foods, Emergency / Emergency Supplies, Alcohol / Alcoholic Beverages, Grocery / Groceries, Baby / Baby Products.

In [6]:
category_map = {'Breakfast Foods': 'Breakfast', 'Emergency Supplies': 'Emergency',
                'Alcoholic Beverages': 'Alcohol', 'Groceries': 'Grocery', 'Baby Products': 'Baby'}
cat['category_clean'] = cat['category_clean'].replace(category_map)
print(cat['category_clean'].value_counts())

category_clean
Emergency        12
Grocery           6
Entertainment     5
Breakfast         4
Electronics       4
Baby              4
Household         4
Alcohol           2
Snacks            2
Health            2
Name: count, dtype: int64


Merged category labels that are different words for the same thing, going from 15 categories down to 10.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [7]:
# TODO
import re

def name_key(name):
    if pd.isna(name):
        return ''
    s = re.sub(r'[^a-z0-9\s]', ' ', str(name).lower())
    return ' '.join(sorted(s.split()))

cat['name_key'] = cat['product_name'].apply(name_key)

sku_counts = cat.groupby('name_key')['sku'].nunique()
fragmented = sku_counts[sku_counts > 1]
print(f"Name groups with more than one SKU: {len(fragmented)}")

candidates = cat[cat['name_key'].isin(fragmented.index)].sort_values('name_key')
candidates[['name_key', 'sku', 'product_name', 'category_clean']]

Name groups with more than one SKU: 14


,name_key,sku,product_name,category_clean
12,24 beer light pack,BEER-24,Light Beer (24-pack),Alcohol
13,24 beer light pack,LTBEER24,Light Beer (24-pack),Alcohol
8,4 aa batteries pack,BAT-AA4,AA Batteries 4-Pack,Emergency
9,4 aa batteries pack,BATT_AA_4PK,AA Batteries 4-Pack,Emergency
30,aid first kit,FIRSTAID-KIT,First Aid Kit,Health
29,aid first kit,FAID-01,First-Aid Kit,Health
15,board game monopoly,MONOPOLY-01,Monopoly Board Game,Entertainment
14,board game monopoly,BG-MONO,Monopoly Board Game,Entertainment
16,board game monopoly,GAME-MONO,Board Game - Monopoly,Entertainment
24,bread white,WHITEBREAD,WHITE BREAD,Grocery


Turned each product name into a normalized key so word order, case, and punctuation stop mattering, then grouped by that key and showed every group that has more than one SKU.


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [8]:
# TODO
assert cat['canonical_sku'].notna().all(), "Mapping produced nulls!"
assert len(cat) == n_rows_original, "Row count changed!"
assert cat['canonical_sku'].nunique() == cat['sku'].nunique() - len(sku_map), "Unexpected SKU count"

print(f"Safe: {len(cat)} rows, no nulls, "
      f"{cat['sku'].nunique()} → {cat['canonical_sku'].nunique()} distinct SKUs")

Safe: 45 rows, no nulls, 45 → 42 distinct SKUs


Asserted that the canonical column has no nulls, the row count didn’t change, and the number of distinct SKUs dropped by exactly the number of variants we mapped.

### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

Bottled water 24-pack: WTR-24 (“BOTTLED WATER 24PK”), WATER24PK (“Bottled Water 24-Pack”), and WTR_24PK (“Water - 24 Pack”). My Q5 key missed all three because “24PK”, “24-Pack”, and “24 Pack” normalize to different words, and one name drops “Bottled.” I’d extend the normalization to standardize pack-size tokens (e.g. turn “24pk” into “24 pack”) before grouping, then map all three to WTR-24.

AA batteries 4-pack: BATT-100 (“BATTERIES AA 4PK”), BAT-AA4, and BATT_AA_4PK (“AA Batteries 4-Pack”). Q5 caught only BAT-AA4 and BATT_AA_4PK, for the same “4PK” reason. I’d group all three under one canonical SKU, but I’d first check package_size, because BATT-100 is listed as a 6 pack and BATT_AA_4PK as a 24 pack even though both names say 4-pack.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

Consolidation stops being data cleaning and becomes a business decision at the point where you cannot determine from the data alone whether two rows represent the exact same product, because merging them then changes inventory and sales numbers and requires judgment. One case I am unsure how to handle is STREAM-01 (“Roku Streaming Stick”) vs ROKU-STD (“Streaming Device”). The shared subcategory and similar prices ($16.70 vs $16.99) suggest they are the same item; however, “Streaming Device” is generic enough that it could be a different model. In this situation, I would contact either the electronics category/merchandising manager or the supplier directly before consolidating these items.